# 🔍 NorthAir Intelligence — 02. Data Quality Analysis
## VIZ-A-THON 2026: Empirical Quality Audit & Anomaly Detection

### Objectives:
1. Missing value distribution per pollutant and meteorological parameter.
2. Station-date duplicate analysis and multiplexing discovery in historical data.
3. Station network temporal completeness (recorded vs expected hours).
4. Statistical distributions (25%, 50%, 75%, 90% percentiles, min, max, mean, std).
5. Sensor clipping & physical boundary auditing (PM2.5 @ 999.99, CO @ 10, BP > 800 mmHg).


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', 35)

sys.path.insert(0, str(Path("../src").resolve()))
from northair.data_loader import DataLoader
from northair.data_profiler import DataProfiler

loader = DataLoader(raw_dir=Path("../data/raw"))
stations_df = loader.load_stations()
aqi_hist_df = loader.load_aqi_history()
aq_2025_df = loader.load_air_quality_hourly_2025()

profiler = DataProfiler(stations_df, aqi_hist_df, aq_2025_df)
print("DataProfiler initialized.")


### 1. Missing Value Audit (2025 Dataset)
CPCB sensors periodically undergo zero/span calibration, maintenance downtime, or telecommunication dropouts.
Let us visualize the missingness percentage across all 23 parameters.


In [ ]:
missing_df = profiler.profile_missing_values(aq_2025_df)
display(missing_df.head(15))

# Plot missing percentages
plt.figure(figsize=(10, 6))
sns.barplot(data=missing_df[missing_df['Missing_Percentage'] > 0], x='Missing_Percentage', y='Column', palette='mako')
plt.title("Missing Value Percentage per Parameter (2025 Hourly Dataset)", fontsize=13, fontweight='bold')
plt.xlabel("Missing Percentage (%)")
plt.xlim(0, 100)
plt.axvline(50, color='red', linestyle='--', label='50% Threshold')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


### 2. Duplicate Audit & Historical Station ID Multiplexing
- Exact duplicate rows: **0** across all 3 files.
- However, in `aqi_history_2017_2023.csv`, `Station ID + Date` has **42,257 duplicates**!
Let us inspect why:


In [ ]:
duplicates = profiler.profile_duplicates()
print("Duplicate Audit Results:")
for ds, metrics in duplicates.items():
    print(f"  • {ds}: {metrics}")

integrity = profiler.profile_station_integrity()
print(f"\nStations with multiple distinct names in historical data: {len(integrity['stations_with_multiple_names_in_history'])}")
for sid, info in integrity['stations_with_multiple_names_in_history'].items():
    print(f"    - {sid}: {info['name_count']} distinct station names (Sample: {info['sample_names'][:2]})")


### 3. Station Temporal Coverage (2025)
In 2025, each station has $365 \times 24 = 8,760$ expected hourly intervals.
Let us assess the completeness of records per station.


In [ ]:
coverage_df = profiler.profile_station_coverage()
print(f"Network Mean Completeness: {coverage_df['Temporal_Coverage_Pct'].mean():.2f}%")
print(f"Highest Coverage: {coverage_df['Temporal_Coverage_Pct'].max()}% ({coverage_df.iloc[0]['Station_Name']})")
print(f"Lowest Coverage: {coverage_df['Temporal_Coverage_Pct'].min()}% ({coverage_df.iloc[-1]['Station_Name']})")

plt.figure(figsize=(10, 4))
sns.histplot(coverage_df['Temporal_Coverage_Pct'], bins=20, kde=True, color='teal')
plt.title("Distribution of Station Temporal Coverage in 2025 (%)", fontsize=13, fontweight='bold')
plt.xlabel("Temporal Coverage (%)")
plt.ylabel("Number of Stations")
plt.tight_layout()
plt.show()


### 4. Numerical Distributions, Percentiles & Anomaly Flags
We calculate complete statistical distribution metrics: Count, Mean, Std, Min, 25%, Median, 75%, 90%, Max, and flag unphysical values or sensor saturation artifacts.


In [ ]:
stats_df, stats_list = profiler.profile_numerical_statistics(aq_2025_df)
display(stats_df[['column', 'count', 'mean', 'median', 'p75', 'p90', 'max', 'flagged_issues']].head(12))


### 5. Distribution of Key Particulate Matter (PM2.5 & PM10)
Let us visualize the distributions of PM2.5 and PM10, noting the CPCB 24-hour National Ambient Air Quality Standards (NAAQS):
- PM2.5 standard: $60 \, \mu\text{g/m}^3$
- PM10 standard: $100 \, \mu\text{g/m}^3$


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PM2.5
sns.histplot(aq_2025_df['PM2.5 (µg/m³)'].dropna(), bins=60, kde=True, ax=axes[0], color='crimson')
axes[0].set_title("PM2.5 Concentration Distribution (2025)", fontweight='bold')
axes[0].set_xlabel("PM2.5 (µg/m³)")
axes[0].axvline(60, color='black', linestyle='--', label='CPCB 24h Standard (60 µg/m³)')
axes[0].axvline(999.99, color='darkred', linestyle=':', label='Sensor Saturation Limit (999.99)')
axes[0].set_xlim(0, 1050)
axes[0].legend()

# PM10
sns.histplot(aq_2025_df['PM10 (µg/m³)'].dropna(), bins=60, kde=True, ax=axes[1], color='darkorange')
axes[1].set_title("PM10 Concentration Distribution (2025)", fontweight='bold')
axes[1].set_xlabel("PM10 (µg/m³)")
axes[1].axvline(100, color='black', linestyle='--', label='CPCB 24h Standard (100 µg/m³)')
axes[1].axvline(1000.0, color='darkred', linestyle=':', label='Sensor Saturation Limit (1000.0)')
axes[1].set_xlim(0, 1050)
axes[1].legend()

plt.tight_layout()
plt.show()


### Summary of Quality Audit Findings:
1. **Saturation Limits**: Hard caps detected at 999.99 for PM2.5 and 1000.0 for PM10.
2. **BP Transducer Drift**: Pressures up to 1069 mmHg indicate uncalibrated sensors.
3. **Historical Multiplexing**: `site_5046` maps to 40 distinct names, requiring station name disambiguation.
4. **No Artificial Imputation**: Missing values preserved as NaN to maintain statistical validity.
